# Part 5: Bring Your Own Data

In [ ]:
# --- Reconnect to Foundry (run this first after opening a new notebook) ---
import subprocess, sys, json, shutil
from pathlib import Path

def run_cmd(args, check=True):
    exe = shutil.which(args[0])
    if exe: args = [exe] + args[1:]
    result = subprocess.run(args, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result

from datetime import date
_user_id = run_cmd(["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"]).stdout.strip()[:6]
SUFFIX = f"{_user_id}-{date.today().strftime('%m%d')}"
RESOURCE_GROUP = f"rg-foundry-workshop-{SUFFIX}"

outputs = json.loads(run_cmd([
    "az", "deployment", "group", "show", "-g", RESOURCE_GROUP, "-n", "main",
    "--query", "properties.outputs", "-o", "json"
]).stdout)

PROJECT_ENDPOINT = outputs["projectEndpoint"]["value"]
MODEL_DEPLOYMENT_NAME = outputs["modelDeploymentName"]["value"]
APP_INSIGHTS_CONN_STR = outputs["appInsightsConnectionString"]["value"]

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool, WebSearchPreviewTool, FunctionTool
from azure.identity import DefaultAzureCredential, AzureCliCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
print(f"✅ Connected to: {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

import asyncio
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient, FoundryAgent
from agent_framework.orchestrations import SequentialBuilder, ConcurrentBuilder, HandoffBuilder, GroupChatBuilder

af_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT_NAME,
    credential=AzureCliCredential(),
)
print("✅ Agent Framework client ready")

# Part 2: Bring Your Own Data (90 min)

Now it's your turn! Use the patterns from Part 1 (notebook 01-04) to build agents with **your own data**.

---
## Section 6: Upload Your Own Data (~20 min)

### Supported file types for `file_search`:
PDF, DOCX, TXT, MD, PPTX, JSON, HTML
### Steps:
1. Place your files in a folder (or use `sample_data/`)
2. Run the upload helper below
3. The vector store ID will be used to create your agent

In [ ]:
# --- File Upload Helper ---
# Upload multiple files to a Foundry vector store.

def upload_files_to_vector_store(file_paths: list, store_name: str = "MyData") -> str:
    """Upload files to a vector store and return the store ID."""
    vs = openai_client.vector_stores.create(name=store_name)
    print(f"📦 Created vector store: {store_name} (ID: {vs.id})")

    for fp in file_paths:
        p = Path(fp)
        if not p.exists():
            print(f"  ⚠️ File not found: {p}")
            continue
        with p.open("rb") as f:
            openai_client.vector_stores.files.upload_and_poll(
                vector_store_id=vs.id, file=f
            )
        print(f"  ✅ Uploaded: {p.name}")

    print(f"\n📦 Vector store ready: {vs.id}")
    return vs.id

print("Helper function defined. Use it in the next cell.")


#### (optional) for CSV/XLSX files, a custom function tool is needed

In [ ]:
# --- CSV / Excel Helper ---
# For CSV/XLSX files, we create a function tool that queries the data.

import pandas as pd

def load_dataframe(path: str) -> pd.DataFrame:
    """Load a CSV or Excel file into a pandas DataFrame."""
    p = Path(path)
    if p.suffix.lower() == ".csv":
        return pd.read_csv(p)
    elif p.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(p)
    else:
        raise ValueError(f"Unsupported format: {p.suffix}")

# This dict holds loaded dataframes — function tools can access them.
user_dataframes = {}

def query_user_data(dataset_name: str, question: str) -> str:
    """Query a loaded dataset. Provide the dataset name and your question."""
    if dataset_name not in user_dataframes:
        return f"Dataset '{dataset_name}' not loaded. Available: {list(user_dataframes.keys())}"
    df = user_dataframes[dataset_name]
    summary = df.describe(include="all").to_string()
    sample = df.head(5).to_string()
    return f"Columns: {list(df.columns)}\nShape: {df.shape}\n\nSummary:\n{summary}\n\nSample rows:\n{sample}"

print("CSV/Excel helper ready. Load your files with:")
print('  user_dataframes["mydata"] = load_dataframe("path/to/file.csv")')


In [ ]:
# ============================================================
# TODO: Upload your files here!
# ============================================================

# Option 1: Upload documents (PDF, DOCX, MD, TXT, PPTX, JSON)
# my_files = [
#     "path/to/your/document.pdf",
#     "path/to/your/report.docx",
# ]
# MY_VECTOR_STORE_ID = upload_files_to_vector_store(my_files, "MyProjectDocs")

# Option 2: Load CSV/Excel data
# user_dataframes["sales"] = load_dataframe("path/to/data.csv")
# print(user_dataframes["sales"].head())

# --- Example using the sample data (uncomment to try) ---
# MY_VECTOR_STORE_ID = upload_files_to_vector_store(
#     [str(Path.cwd() / "sample_data" / "product_spec.md")],
#     "SampleDocs"
# )
# user_dataframes["sales"] = load_dataframe(str(Path.cwd() / "sample_data" / "sales_data.csv"))

print("👆 Uncomment and modify the code above to upload your files.")

---
## Section 7: Build Your Custom Agent (~25 min)

Use what you learned in Sections 1–5 to create an agent tailored to **your data**.

**Tips from Section 3:**
- Define a clear **persona** and **expertise area**
- Set **constraints** (what it should/shouldn't do)
- Specify **output format** (JSON, markdown, bullet points)
- Add **few-shot examples** if you want a specific style

In [ ]:
# ============================================================
# TODO: Build your custom agent!
# ============================================================

# TODO: Choose your agent name
MY_AGENT_NAME = "my-custom-agent"  # e.g. "rfp-assistant", "data-analyst"

# TODO: Write your instructions (use the patterns from Section 3)
MY_INSTRUCTIONS = (
    "You are a helpful assistant. "  # <-- Replace with your persona and rules
    "Answer questions based on the provided documents."
)

# TODO: Choose your tools
my_tools = []
# Uncomment the tools you want:
my_tools.append(FileSearchTool(vector_store_ids=[MY_VECTOR_STORE_ID]))  # Document search
my_tools.append(WebSearchPreviewTool())  # Web search

# If using CSV data, add the function tool:
# csv_tool = FunctionTool([query_user_data])
# csv_ts = ToolSet()
# csv_ts.add(csv_tool)
# (pass toolset=csv_ts to create_version instead of tools=my_tools)

my_agent = project_client.agents.create_version(
    agent_name=MY_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=MY_INSTRUCTIONS,
        tools=my_tools if my_tools else None,
    ),
)

print(f"✅ Custom agent created: {my_agent.name} v{my_agent.version}")


In [ ]:
# --- Test Your Agent ---

# TODO: Replace these with questions relevant to YOUR data
test_questions = [
    "What is the main topic of the uploaded documents?",
    "Summarize the key findings.",
    "What are the top 3 recommendations?",
]

for i, q in enumerate(test_questions, 1):
    print(f"\n{'=' * 50}")
    print(f"Test {i}: {q}")
    print("=" * 50)

    response = openai_client.responses.create(
        input=q,
        extra_body={"agent_reference": {"name": my_agent.name, "type": "agent_reference"}},
    )
    print(f"🤖 {response.output_text[:500]}")


---
## Section 8: Build Your Multi-Agent Workflow (~30 min)

Combine multiple agents into a workflow. Choose the pattern that fits your use case:

| Your Goal | Pattern | Example |
|-----------|---------|----------|
| Step-by-step processing | **Sequential** | Research → Draft → Review |
| Multiple perspectives at once | **Concurrent** | Legal + Technical + Business analysis |
| Route to the right expert | **Handoff** | Triage → Specialist agents |
| Collaborative discussion | **Group Chat** | Team of agents refining an answer |
| Highly complex conditions | **Custom Workflow** | Team of agents with mixed pattern and conditions |

In [ ]:
# ============================================================
# TODO: Define your specialist agents
# ============================================================

# Define 2-3 agents that work together on your use case.
# Use the same af_client from Section 5.

agent_1 = Agent(
    client=af_client, name="Agent1",
    instructions="TODO: Define this agent's role and expertise.",
)

agent_2 = Agent(
    client=af_client, name="Agent2",
    instructions="TODO: Define this agent's role and expertise.",
)

# Optional third agent:
# agent_3 = Agent(
#     client=af_client, name="Agent3",
#     instructions="TODO: Define this agent's role and expertise.",
# )

print("Specialist agents defined. Choose your orchestration pattern in the next cell.")

In [ ]:
# ============================================================
# TODO: Choose your orchestration pattern
# Uncomment ONE of the blocks below.
# ============================================================

MY_TASK = "TODO: Describe the task for your agents"  # <-- Change this!

# --- Option A: Sequential ---
# workflow = SequentialBuilder(participants=[agent_1, agent_2]).build()

# --- Option B: Concurrent ---
# workflow = ConcurrentBuilder(participants=[agent_1, agent_2]).build()

# --- Option C: Handoff ---
# for a in [agent_1, agent_2]:
#     a.require_per_service_call_history_persistence = True
# router = Agent(
#     client=af_client, name="Router",
#     instructions="Route to the right specialist based on the request.",
# )
# router.require_per_service_call_history_persistence = True
# workflow = (
#     HandoffBuilder(name="my-workflow", participants=[router, agent_1, agent_2])
#     .with_start_agent(router)
#     .add_handoff(router, [agent_1, agent_2])
#     .add_handoff(agent_1, [router])
#     .add_handoff(agent_2, [router])
#     .build()
# )

# --- Option D: Group Chat ---
# orch = Agent(
#     client=af_client, name="Orchestrator",
#     instructions="Coordinate the discussion between agents.",
# )
# workflow = (
#     GroupChatBuilder(
#         participants=[agent_1, agent_2],
#         orchestrator_agent=orch,
#         max_rounds=6,
#     ).build()
# )

print("👆 Uncomment your chosen pattern above, then run the next cell.")

In [ ]:
# --- Run Your Workflow ---

my_workflow_agent = workflow.as_agent()

result = await my_workflow_agent.run(MY_TASK)
if result.messages:
    for i, msg in enumerate(result.messages, 1):
        name = msg.author_name or msg.role
        if msg.text:
            print(f"\n[{i:02d} {name}]:")
            print(msg.text[:500])
            print("---")

print("\n✅ Custom workflow complete! Check Foundry portal → Tracing for the workflow spans.")


---
## Section 9: Wrap-Up & Next Steps

### 🎉 Congratulations!

You've built and tested:
- ✅ Single agents with function tools, web search, and file search
- ✅ Prompt-engineered agents with structured output
- ✅ Multi-step reasoning agents
- ✅ Agent quality evaluation with built-in evaluators
- ✅ Multi-agent workflows (sequential, concurrent, handoff, group chat)
- ✅ Custom agents with your own data

### 🚀 Your agents are still live!

All agents you created during this workshop are **still available** in your Foundry project.
You can continue building on them:

| Agent | Type | How to find it |
|-------|------|----------------|
| workshop-assistant | Prompt Agent | Foundry Portal → Agents |
| pricing-assistant | Function Tool Agent | Foundry Portal → Agents |
| web-researcher | Web Search Agent | Foundry Portal → Agents |
| document-expert | File Search Agent | Foundry Portal → Agents |
| sales-analyst | CSV Tool Agent | Foundry Portal → Agents |
| requirements-analyst | Multi-Step Agent | Foundry Portal → Agents |
| Your custom agent(s) | Custom | Foundry Portal → Agents |

### 📊 Review Your Traces & Evaluation

- **Traces**: [Foundry Portal](https://ai.azure.com) → your project → **Tracing**
- **Evaluation**: [Foundry Portal](https://ai.azure.com) → your project → **Evaluation**

### 📚 Next Steps

| Topic | Resource |
|-------|----------|
| Deploy as hosted agent | [Hosted Agents docs](https://learn.microsoft.com/azure/ai-foundry/agents/concepts/hosted-agents) |
| Production RAG with AI Search | [AI Search tool](https://learn.microsoft.com/azure/ai-foundry/agents/how-to/tools/ai-search) |
| Governance & monitoring | [Enterprise Agent Tutorial](https://github.com/microsoft-foundry/foundry-samples/tree/main/samples/python/enterprise-agent-tutorial) |
| Agent Framework SDK | [GitHub repo](https://github.com/microsoft/agent-framework) |
| Evaluation best practices | [Azure AI Evaluation](https://learn.microsoft.com/azure/ai-foundry/evaluation/) |

In [ ]:
# ============================================================
# OPTIONAL: Resource Cleanup
# ============================================================
# Uncomment the lines below to delete ALL Azure resources
# created during this workshop. This is irreversible!

# print(f"⚠️  Deleting resource group: {RESOURCE_GROUP}")
# run_cmd(["az", "group", "delete", "-n", RESOURCE_GROUP, "--yes", "--no-wait"])
# print("✅ Deletion initiated (will complete in background)")

print("No cleanup performed — your agents and resources are still available.")
print(f"Resource group: {RESOURCE_GROUP}")
print(f"Project endpoint: {PROJECT_ENDPOINT}")